# Phase 1.3: Text Statistics

Use the IMDB reviews to measure word frequencies, N-grams, lexical diversity, review lengths, and Shannon entropy. Complete the exercises in order and explain each result in your own words.

## Learning goals

By the end of this notebook, you should be able to:

- build and interpret a corpus frequency distribution;
- implement and count contiguous N-grams;
- calculate TTR and hapax/dis-legomena proportions;
- summarize document lengths with mean, median, and variance;
- calculate and interpret Shannon entropy; and
- explain how preprocessing decisions change text statistics.

## 1. Setup

In [1]:
import sys
from pathlib import Path

import pandas as pd
import nltk
from nltk.tokenize import word_tokenize

# Reuse the preprocessing pipeline implemented in part 02.
notebook_dir = Path.cwd()
if notebook_dir.name != "notebooks":
    notebook_dir = Path(
        "phase-1-machine-learning-nlp/03-text-statistics/notebooks"
    ).resolve()

part_2_dir = (notebook_dir / "../../02-preprocessing-tokenization").resolve()
part_2_src = part_2_dir / "src"
if str(part_2_src) not in sys.path:
    sys.path.insert(0, str(part_2_src))

from preprocessor import STOP_WORDS, preprocess_text

imdb_csv_path = part_2_dir / "data/IMDB Dataset.csv"


## 2. Load and prepare the corpus

Load the review column from the IMDB CSV. Apply one consistent preprocessing and tokenization pipeline to every review. Keep reviews separate so review-length and N-gram calculations do not accidentally cross document boundaries.

In [2]:
# Exercise: Load the CSV and validate that the review column exists.
# Exercise: Check for missing.
# Clean and tokenize each review consistently.
# Exercise: Store the result as one token list per review.
# Exercise: Display a few before/after examples and the number of reviews.

df = pd.read_csv(imdb_csv_path)
print("Null Values:\n",df.isnull().sum(),sep="")

df["Cleaned"] = df["review"].apply(
    lambda review: preprocess_text(
        review,
        remove_stopwords=True,
        apply_stemming=False,
    )
)
df["Tokenized"] = df["Cleaned"].apply(
    lambda text: [
        token for token in word_tokenize(text)
        if token.lower() not in STOP_WORDS
    ]
)

print(df.head())

Null Values:
review       0
sentiment    0
dtype: int64
                                              review  ...                                          Tokenized
0  One of the other reviewers has mentioned that ...  ...  [one, reviewers, mentioned, watching, oz, epis...
1  A wonderful little production. <br /><br />The...  ...  [wonderful, little, production, filming, techn...
2  I thought this was a wonderful way to spend ti...  ...  [thought, wonderful, way, spend, time, hot, su...
3  Basically there's a family where a little boy ...  ...  [basically, theres, family, little, boy, jake,...
4  Petter Mattei's "Love in the Time of Money" is...  ...  [petter, matteis, love, time, money, visually,...

[5 rows x 4 columns]


## 3. Word-frequency

In [3]:
# Exercise: Build one corpus-level frequency distribution.
# Exercise: Calculate the total token count and unique-token count.
# Exercise: Find and clearly display the 10 most frequent words and counts.
# Exercise: Add sanity checks that connect the Counter totals to your tokens.

word_freq = {}

for review in df["Tokenized"]:
    for token in review:
        word_freq[token] = word_freq.get(token, 0) + 1



In [4]:
print("Unique tokens count:", len(word_freq.items()))
print("Most frequent token:", max(word_freq, key=word_freq.get))
print("\n")
sorted_keys = sorted(word_freq, key=word_freq.get, reverse=True)

for i in range(10):
    print(i+1,"'",sorted_keys[i],"'",word_freq[sorted_keys[i]])

Unique tokens count: 214773
Most frequent token: movie


1 ' movie ' 83576
2 ' film ' 74512
3 ' one ' 50391
4 ' like ' 38834
5 ' good ' 28502
6 ' even ' 24285
7 ' would ' 24001
8 ' time ' 23297
9 ' really ' 22900
10 ' see ' 22437


### Frequency observations

Record what you saw.

### My observation: the most frequent words (tokens) on a corpus that is not preprocessed are stop words and punctuation. After cleaning the tokens we can see that the most frequent word is movies which makes sense because the dataset is movie reviews.

## 4. N-grams

Define an N-gram.

#### My Answer: N-grams are a way of featurizing our tokens so N represents the number of token grouping together for better context and to calculate statistics of what token comes before each token and which come after. So assuming bigrams if we had the sentence ' We are here at the cinema ' it would be (if tokens are words) [We are, are here, here at, at the, the cinema] it also depends on preprocessing like removing stop words but the idea is there are many ways to do this there unigrams, bigrams, trigrams,... the difference is how much context each one adds and is it too much or too little. 

In [5]:
def extract_ngrams(tokens, n):
    if n <= 0:
        raise ValueError("n must be a positive integer")

    if len(tokens) < n:
        return []

    ngrams = []
    for i in range(len(tokens) - n + 1):
        ngrams.append(tuple(tokens[i:i+n]))
    return ngrams

In [6]:
from collections import Counter

# Find and clearly display the top 10 N-grams for n=2, n=3, and n=4.
# Each review is processed separately so N-grams never cross review boundaries.
top_ngrams_by_n = {}

for n in (2, 3, 4):
    ngram_counts = Counter()

    for review_tokens in df["Tokenized"]:
        ngram_counts.update(extract_ngrams(review_tokens, n))

    expected_total = sum(
        max(len(review_tokens) - n + 1, 0)
        for review_tokens in df["Tokenized"]
    )
    assert sum(ngram_counts.values()) == expected_total

    top_ngrams_by_n[n] = ngram_counts.most_common(10)

    print(f"Top 10 {n}-grams")
    for rank, (ngram, count) in enumerate(top_ngrams_by_n[n], start=1):
        print(f"{rank:>2}. {' '.join(ngram)}: {count:,}")
    print()


Top 10 2-grams
 1. ever seen: 2,533
 2. ive seen: 2,148
 3. special effects: 2,134
 4. dont know: 2,054
 5. even though: 1,868
 6. one best: 1,836
 7. looks like: 1,622
 8. much better: 1,424
 9. waste time: 1,420
10. see movie: 1,400

Top 10 3-grams
 1. ive ever seen: 988
 2. worst movie ever: 360
 3. dont waste time: 325
 4. one worst movies: 312
 5. movie ever seen: 299
 6. new york city: 252
 7. dont get wrong: 240
 8. movies ever seen: 214
 9. world war ii: 209
10. worst movies ever: 207

Top 10 4-grams
 1. one worst movies ever: 174
 2. movie ive ever seen: 167
 3. worst movie ever seen: 156
 4. movies ive ever seen: 138
 5. worst movies ever seen: 104
 6. worst movie ive ever: 93
 7. one worst films ever: 93
 8. one worst movies ive: 88
 9. films ive ever seen: 81
10. ive seen long time: 80



### N-gram observations

Compare the top bigrams, trigrams, and 4-grams. Which sequences appear meaningful, and which are mostly explained by common grammar or preprocessing choices?

## 5. Lexical-diversity statistics

Write the formula and meaning of each measure before implementing it:

- **Type-Token Ratio (TTR)**
- **Hapax legomena**: types occurring exactly once
- **Dis legomena**: types occurring exactly twice

State whether your hapax and dis-legomena proportions use total types or total tokens as the denominator. Explain why.

### Formulas and meanings

- **Type-Token Ratio:** `TTR = unique tokens / total tokens`. It measures lexical variety; a larger value means a greater share of the corpus consists of different words.
- **Hapax legomena:** unique tokens that occur exactly once. `Hapax proportion = hapax types / total types`.
- **Dis legomena:** unique tokens that occur exactly twice. `Dis-legomena proportion = dis types / total types`.

The hapax and dis-legomena proportions use **total types (unique tokens)** as their denominator because both measures describe the composition of the vocabulary. Using total tokens would instead measure how much of the entire corpus is contributed by rare words.

In [7]:
# These frequencies were built from the cleaned tokens in df["Tokenized"].
clean_token_frequencies = word_freq

total_tokens = sum(clean_token_frequencies.values())
total_types = len(clean_token_frequencies)

hapax_count = sum(
    frequency == 1 for frequency in clean_token_frequencies.values()
)
dis_legomena_count = sum(
    frequency == 2 for frequency in clean_token_frequencies.values()
)

# Return 0.0 for an empty corpus instead of dividing by zero.
ttr = total_types / total_tokens if total_tokens else 0.0
hapax_proportion = hapax_count / total_types if total_types else 0.0
dis_legomena_proportion = (
    dis_legomena_count / total_types if total_types else 0.0
)

# Confirm that the frequency total matches the cleaned token lists.
assert total_tokens == sum(len(tokens) for tokens in df["Tokenized"])

print(f"Total tokens: {total_tokens:,}")
print(f"Total types (unique tokens): {total_types:,}")
print(f"Type-Token Ratio (TTR): {ttr:.6f}")
print(f"Hapax legomena count: {hapax_count:,}")
print(f"Hapax proportion (of types): {hapax_proportion:.6f}")
print(f"Dis legomena count: {dis_legomena_count:,}")
print(f"Dis-legomena proportion (of types): {dis_legomena_proportion:.6f}")

Total tokens: 5,929,057
Total types (unique tokens): 214,773
Type-Token Ratio (TTR): 0.036224
Hapax legomena count: 133,263
Hapax proportion (of types): 0.620483
Dis legomena count: 21,654
Dis-legomena proportion (of types): 0.100823


## 6. Review-length statistics

Use the number of processed tokens in each retained review as its length.

- The **mean** is the total number of cleaned tokens divided by the number of reviews. It describes the average review length but is affected by unusually long reviews.
- The **median** is the middle review length after sorting. It describes a typical review more robustly when lengths are skewed.
- The **variance** is the average squared distance from the mean. It measures how widely review lengths are spread. This analysis uses population variance because all 50,000 reviews in the analyzed corpus are included.

In [8]:
from statistics import fmean, median, pvariance

# A review's length is its number of cleaned tokens.
review_lengths = [len(tokens) for tokens in df["Tokenized"]]

if not review_lengths:
    raise ValueError("Cannot calculate review-length statistics for an empty corpus")

mean_review_length = fmean(review_lengths)
median_review_length = median(review_lengths)
review_length_variance = pvariance(review_lengths)

# Population variance is used because all reviews in the analyzed corpus are included.
print(f"Number of reviews: {len(review_lengths):,}")
print(f"Mean words per review: {mean_review_length:.2f}")
print(f"Median words per review: {median_review_length:.2f}")
print(f"Population variance: {review_length_variance:.2f}")
print(f"Minimum words in a review: {min(review_lengths):,}")
print(f"Maximum words in a review: {max(review_lengths):,}")

Number of reviews: 50,000
Mean words per review: 118.58
Median words per review: 88.00
Population variance: 7965.46
Minimum words in a review: 3
Maximum words in a review: 1,420


## 7. Shannon entropy

For a word distribution, Shannon entropy is `H = -Σ p(w) log₂ p(w)`, where `p(w)` is a word's count divided by the total token count. Base 2 is used, so entropy is measured in **bits**. Lower entropy means probability is concentrated among fewer words; higher entropy means it is distributed more evenly across the vocabulary.

In [ ]:
import math

total_word_count = sum(word_freq.values())
vocabulary_size = len(word_freq)

if total_word_count == 0:
    word_probabilities = {}
    probability_sum = 0.0
    entropy_bits = 0.0
else:
    word_probabilities = {
        word: count / total_word_count
        for word, count in word_freq.items()
    }
    probability_sum = sum(word_probabilities.values())

    # Implement H = -sum(p * log2(p)) directly.
    entropy_bits = -sum(
        probability * math.log2(probability)
        for probability in word_probabilities.values()
    )

maximum_entropy_bits = (
    math.log2(vocabulary_size) if vocabulary_size > 1 else 0.0
)


print(f"Total probability: {probability_sum:.12f}")
print(f"Corpus Shannon entropy: {entropy_bits:.6f} bits")
print(f"Maximum entropy for {vocabulary_size:,} words: {maximum_entropy_bits:.6f} bits")

Total probability: 1.000000000000
Corpus Shannon entropy: 12.602946 bits
Maximum entropy for 214,773 words: 17.712453 bits
Normalized entropy: 0.711530
